In [1]:
import gensim.downloader

In [2]:
models = gensim.downloader.info()['models'].keys()
print("Modelos disponíveis no Gensim:")
for model in models:
    print(model)

Modelos disponíveis no Gensim:
fasttext-wiki-news-subwords-300
conceptnet-numberbatch-17-06-300
word2vec-ruscorpora-300
word2vec-google-news-300
glove-wiki-gigaword-50
glove-wiki-gigaword-100
glove-wiki-gigaword-200
glove-wiki-gigaword-300
glove-twitter-25
glove-twitter-50
glove-twitter-100
glove-twitter-200
__testing_word2vec-matrix-synopsis


In [3]:
model = gensim.downloader.load('glove-wiki-gigaword-50')

[==================================================] 100.0% 66.0/66.0MB downloaded


In [4]:
print(model['tower'])

[ 1.1474e+00  1.1811e+00  7.4556e-01 -5.9101e-02  5.0499e-01 -7.0449e-01
 -3.2136e-01 -4.5390e-01 -4.5763e-01 -7.5341e-01 -3.3511e-01 -2.4975e-02
 -5.0192e-01  6.3773e-01 -8.3059e-01  8.3565e-01 -2.4701e-01  3.2421e-01
 -1.1103e+00 -2.1335e-02  6.8717e-01 -3.9340e-01 -1.6390e+00 -5.0493e-01
 -1.6684e-01 -6.7649e-01 -3.1798e-01  8.8503e-01 -3.1552e-02 -1.5608e-01
  1.9805e+00 -1.1870e+00  8.3342e-01 -1.8369e-01 -2.6691e-01  1.1619e-01
  1.1023e+00 -3.5937e-01  2.5015e-02 -4.0615e-02  3.0681e-01 -4.1076e-01
  8.4586e-02  2.2475e-01 -5.0955e-01  6.5819e-01 -1.2432e-01 -1.4039e+00
  1.6178e-04 -5.2529e-01]


In [5]:
tower_vector = model['tower']
roof_vector = model['roof']
similarity = model.similarity('tower', 'roof')
print(f"Similaridade entre 'tower' e 'roof': {similarity}")

Similaridade entre 'tower' e 'roof': 0.7780845165252686


In [6]:
tower_vector = model['tower']
pencil_vector = model['pencil']
similarity = model.similarity('tower', 'pencil')
print(f"Similaridade entre 'tower' e 'pencil': {similarity}")

Similaridade entre 'tower' e 'pencil': 0.1350826621055603


# OpenAI Chat Completion Demo

## Demonstração de predição de próxima palavra e efeito da temperatura

In [8]:
# Importar bibliotecas
import os
from openai import OpenAI
from dotenv import load_dotenv

# Carregar variáveis do arquivo .env
load_dotenv()

# Inicializar cliente OpenAI
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print("✓ OpenAI client inicializado com sucesso!")

✓ OpenAI client inicializado com sucesso!


## Demo 1: Iterando sobre próximas palavras

Vamos gerar texto palavra por palavra, mostrando como o modelo prevê a próxima palavra iterativamente.

In [ ]:
def predict_next_words(prompt, num_iterations=5, temperature=0.7, max_tokens=5):
    """
    Gera texto iterativamente, palavra por palavra
    """
    print(f"📝 Prompt inicial: '{prompt}'")
    print(f"🌡️  Temperature: {temperature}")
    print("="*70)
    
    current_text = prompt
    
    for i in range(num_iterations):
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",  # Modelo simples e econômico
            messages=[
                {"role": "system", "content": "You are a helpful assistant. Continue the text naturally."},
                {"role": "user", "content": f"Continue this text with a few words: {current_text}"}
            ],
            temperature=temperature,
            max_tokens=max_tokens
        )
        
        # Obter o texto gerado
        generated_text = response.choices[0].message.content.strip()
        
        # Mostrar estatísticas da predição
        print(f"\n🔄 Iteração {i+1}:")
        print(f"   Texto atual: '{current_text}'")
        print(f"   Próximas palavras: '{generated_text}'")
        print(f"   Finish reason: {response.choices[0].finish_reason}")
        print(f"   Tokens usados: {response.usage.total_tokens}")
        
        # Atualizar texto atual
        current_text += " " + generated_text
    
    print("\n" + "="*70)
    print(f"✨ Texto final: '{current_text}'")
    print("="*70)
    
    return current_text

# Exemplo de uso
prompt = "The future of artificial intelligence"
result = predict_next_words(prompt, num_iterations=5, temperature=0.7, max_tokens=5)

📝 Prompt inicial: 'The future of artificial intelligence'
🌡️  Temperature: 0.7

🔄 Iteração 1:
   Texto atual: 'The future of artificial intelligence'
   Próximas palavras: 'is an exciting and rapidly'
   Finish reason: length
   Tokens usados: 40

🔄 Iteração 1:
   Texto atual: 'The future of artificial intelligence'
   Próximas palavras: 'is an exciting and rapidly'
   Finish reason: length
   Tokens usados: 40

🔄 Iteração 2:
   Texto atual: 'The future of artificial intelligence is an exciting and rapidly'
   Próximas palavras: 'evolving field that has'
   Finish reason: length
   Tokens usados: 45

🔄 Iteração 2:
   Texto atual: 'The future of artificial intelligence is an exciting and rapidly'
   Próximas palavras: 'evolving field that has'
   Finish reason: length
   Tokens usados: 45

🔄 Iteração 3:
   Texto atual: 'The future of artificial intelligence is an exciting and rapidly evolving field that has'
   Próximas palavras: 'captured the attention of'
   Finish reason: length
   T

## Demo 2: Efeito da Temperatura

A temperatura controla a aleatoriedade das predições:
- **Temperatura baixa (0.0-0.3)**: Respostas mais determinísticas e previsíveis
- **Temperatura média (0.5-0.7)**: Equilíbrio entre criatividade e coerência
- **Temperatura alta (0.8-2.0)**: Respostas mais criativas e aleatórias

Vamos comparar diferentes temperaturas com o mesmo prompt!

In [ ]:
def compare_temperatures(prompt, temperatures=[0.0, 0.5, 1.0, 1.5], max_tokens=50):
    """
    Compara o efeito de diferentes temperaturas na geração de texto
    """
    print(f"📝 Prompt: '{prompt}'")
    print("="*70)
    
    results = []
    
    for temp in temperatures:
        print(f"\n🌡️  TEMPERATURA: {temp}")
        print("-"*70)
        
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": prompt}
            ],
            temperature=temp,
            max_tokens=max_tokens
        )
        
        generated_text = response.choices[0].message.content.strip()
        
        # Mostrar informações detalhadas
        print(f"📄 Resposta: {generated_text}")
        print(f"⚙️  Finish reason: {response.choices[0].finish_reason}")
        print(f"🔢 Tokens: prompt={response.usage.prompt_tokens}, "
              f"completion={response.usage.completion_tokens}, "
              f"total={response.usage.total_tokens}")
        
        results.append({
            'temperature': temp,
            'text': generated_text,
            'tokens': response.usage.total_tokens
        })
    
    print("\n" + "="*70)
    print("📊 RESUMO COMPARATIVO:")
    print("="*70)
    for i, res in enumerate(results, 1):
        print(f"\n{i}. Temp={res['temperature']}: {res['text'][:80]}...")
    
    return results

# Exemplo de uso
prompt = "Write a short story about a robot learning to"
results = compare_temperatures(prompt, temperatures=[0.0, 0.7, 1.5], max_tokens=40)

## Demo 3: Múltiplas gerações com mesma temperatura

Executando várias vezes com a mesma temperatura para ver a variação nas respostas:

In [ ]:
def multiple_generations(prompt, temperature=0.7, num_runs=3, max_tokens=30):
    """
    Gera múltiplas respostas com a mesma temperatura para demonstrar variabilidade
    """
    print(f"📝 Prompt: '{prompt}'")
    print(f"🌡️  Temperature: {temperature}")
    print(f"🔁 Número de execuções: {num_runs}")
    print("="*70)
    
    for i in range(num_runs):
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "You are a creative assistant."},
                {"role": "user", "content": prompt}
            ],
            temperature=temperature,
            max_tokens=max_tokens
        )
        
        generated_text = response.choices[0].message.content.strip()
        
        print(f"\n🎲 Execução {i+1}:")
        print(f"   {generated_text}")
        print(f"   (Tokens: {response.usage.total_tokens})")
    
    print("\n" + "="*70)

# Teste com temperatura baixa (mais determinístico)
print("🔹 TEMPERATURA BAIXA (0.2) - Mais previsível:\n")
multiple_generations(
    "Complete the sentence: Machine learning is",
    temperature=0.2,
    num_runs=3,
    max_tokens=25
)

print("\n\n")

# Teste com temperatura alta (mais criativo)
print("🔸 TEMPERATURA ALTA (1.5) - Mais variado:\n")
multiple_generations(
    "Complete the sentence: Machine learning is",
    temperature=1.5,
    num_runs=3,
    max_tokens=25
)

## Demo 4: Visualizando Predições com Top Logprobs

O OpenAI pode retornar as probabilidades das palavras mais prováveis (top logprobs), permitindo ver as alternativas que o modelo considerou:

In [ ]:
import math

def show_token_predictions(prompt, temperature=0.7, max_tokens=10):
    """
    Mostra as predições de tokens com suas probabilidades
    """
    print(f"📝 Prompt: '{prompt}'")
    print(f"🌡️  Temperature: {temperature}")
    print("="*70)
    
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=temperature,
        max_tokens=max_tokens,
        logprobs=True,  # Habilitar logprobs
        top_logprobs=3  # Mostrar top 3 alternativas para cada token
    )
    
    generated_text = response.choices[0].message.content.strip()
    print(f"\n✨ Texto gerado: '{generated_text}'")
    print("\n" + "="*70)
    print("📊 ANÁLISE TOKEN POR TOKEN:")
    print("="*70)
    
    # Analisar cada token gerado
    logprobs_data = response.choices[0].logprobs.content
    
    for i, token_data in enumerate(logprobs_data):
        token = token_data.token
        logprob = token_data.logprob
        probability = math.exp(logprob) * 100  # Converter log-prob para porcentagem
        
        print(f"\n🔤 Token {i+1}: '{token}'")
        print(f"   Probabilidade: {probability:.2f}%")
        
        # Mostrar alternativas consideradas
        if token_data.top_logprobs:
            print(f"   📋 Top alternativas:")
            for alt in token_data.top_logprobs:
                alt_prob = math.exp(alt.logprob) * 100
                print(f"      • '{alt.token}' - {alt_prob:.2f}%")
    
    print("\n" + "="*70)
    
    return generated_text

# Exemplo de uso
result = show_token_predictions(
    "The capital of France is",
    temperature=0.3,
    max_tokens=5
)

📝 Prompt: 'The capital of France is'
🌡️  Temperature: 0.3

✨ Texto gerado: 'Paris.'

📊 ANÁLISE TOKEN POR TOKEN:

🔤 Token 1: 'Paris'
   Probabilidade: 87.12%
   📋 Top alternativas:
      • 'Paris' - 87.12%
      • ' Paris' - 12.85%
      • ' ' - 0.01%

🔤 Token 2: '.'
   Probabilidade: 60.61%
   📋 Top alternativas:
      • '.' - 60.61%
      • '<|end|>' - 39.38%
      • '!' - 0.01%


✨ Texto gerado: 'Paris.'

📊 ANÁLISE TOKEN POR TOKEN:

🔤 Token 1: 'Paris'
   Probabilidade: 87.12%
   📋 Top alternativas:
      • 'Paris' - 87.12%
      • ' Paris' - 12.85%
      • ' ' - 0.01%

🔤 Token 2: '.'
   Probabilidade: 60.61%
   📋 Top alternativas:
      • '.' - 60.61%
      • '<|end|>' - 39.38%
      • '!' - 0.01%

